BARRIDO DE FONDOS CONCURSABLES PARA EL AÑO 2026

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib3

In [2]:
# Apagamos las advertencias molestas de seguridad
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Iniciando Radar de Vigilancia Tecnológica 2026...")

Iniciando Radar de Vigilancia Tecnológica 2026...


In [3]:
# 1. NUESTROS OBJETIVOS (Excluyendo el gobierno regional)
sitios_objetivo = {
    "ANID (Ciencia y Transferencia)": "https://anid.cl/concursos/",
    "CORFO (Innovación y Startups)": "https://www.corfo.cl/sites/cpp/convocatorias"
}

# 2. FILTRO LÁSER (Palabras clave para tus proyectos de hardware, IA y astronomía)
palabras_clave = [
    "tecnológica", "innovación", "transferencia", "astrofísica",
    "emprendimiento", "aplicada", "hardware", "software", "startup", "ia", "desafíos"
]

# 3. EL DISFRAZ (Para evitar bloqueos de firewall)
headers_avanzados = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-CL,es;q=0.9,en-US;q=0.8,en;q=0.7"
}

fondos_encontrados = []

Las páginas de tecnología a veces esconden sus datos detrás de menús interactivos que cargan después de que abres la página (usando JavaScript).

requests y BeautifulSoup son rapidísimos, pero solo leen el texto estático inicial.

Por lo que si se quiere generar una busqueda mucho más detallada se debe generar una mejora en el codigo anteriro. Además, es importante señalar que si el código de arriba dice "Extracción exitosa en CORFO" pero te trae resultados vacíos o incompletos, significa que la página es dinámica y cuando se llegue a ese punto, el siguiente salto evolutivo para la programación debe ser reemplazar requests por Selenium.

Selenium abre un navegador fantasma en tu computador, el cual permite hacer clic en botones, esperar a que carguen las animaciones para luego obtener los datos.

In [4]:
# 4. EL MOTOR DE EXTRACCIÓN
for nombre_sitio, url in sitios_objetivo.items():
    print(f"\nExplorando: {nombre_sitio}...")

    try:
        # Hacemos la petición saltando proxies
        respuesta = requests.get(url, headers=headers_avanzados, proxies={"http": None, "https": None}, verify=False, timeout=15)

        if respuesta.status_code != 200:
            print(f"  [!] Código {respuesta.status_code} al intentar leer {url}")
            continue

        sopa = BeautifulSoup(respuesta.text, 'html.parser')

        # --- LÓGICA DE LECTURA PARA ANID ---
        if "anid.cl" in url:
            # ANID suele agrupar los concursos en filas o listas
            concursos = sopa.find_all(['tr', 'div'], class_=lambda c: c and 'concurso' in c.lower())
            if not concursos: # Plan B si cambian la clase
                concursos = sopa.find_all('article')

            for item in concursos:
                texto = item.text.strip().replace('\n', ' ')
                if len(texto) > 15:
                    enlace_tag = item.find('a', href=True)
                    link = enlace_tag['href'] if enlace_tag else url

                    fondos_encontrados.append({
                        "Fuente": "ANID",
                        "Nombre de la Convocatoria": texto[:120] + "...",
                        "Enlace Oficial": link
                    })

        # --- LÓGICA DE LECTURA PARA CORFO ---
        elif "corfo.cl" in url:
            # CORFO usa un sistema de tarjetas (cards) para sus programas
            tarjetas = sopa.find_all('div', class_=lambda c: c and 'card' in c.lower())

            for tarjeta in tarjetas:
                titulo_tag = tarjeta.find(['h2', 'h3', 'h4'])
                titulo = titulo_tag.text.strip() if titulo_tag else "Convocatoria CORFO"

                enlace_tag = tarjeta.find('a', href=True)
                link = enlace_tag['href'] if enlace_tag else url
                # CORFO a veces usa links relativos, los completamos:
                if link.startswith('/'): link = "https://www.corfo.cl" + link

                fondos_encontrados.append({
                    "Fuente": "CORFO",
                    "Nombre de la Convocatoria": titulo,
                    "Enlace Oficial": link
                })

        print(f"  -> Extracción exitosa en {nombre_sitio}.")

    except Exception as e:
        print(f"  [X] Error crítico escrapeando {nombre_sitio}: {e}")


Explorando: ANID (Ciencia y Transferencia)...
  -> Extracción exitosa en ANID (Ciencia y Transferencia).

Explorando: CORFO (Innovación y Startups)...
  [!] Código 404 al intentar leer https://www.corfo.cl/sites/cpp/convocatorias
